# Pilot inter-annotator agreement

Computes Cohen's and Fleiss' kappa across the three annotators on the 100-review pilot set (guidelines v1). Expects `Annotation_Arpita.xlsx`, `Annotation_Harsh.xlsx`, `Annotation_Punit.xlsx` -- the same 100 reviews, labelled independently.

## Setup

In [ ]:
!pip install statsmodels -q

## Load annotation files

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## Agreement metrics
Category-detection agreement, category+sentiment agreement, sentiment-only agreement (restricted to aspects all three annotators flagged), and per-category Fleiss' kappa.

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

CATS = ["Delivery", "Price_Charges", "Product_Quality", "Wrong_Missing_Item", "Refund_Return",
        "Payment_Wallet", "Customer_Support", "App_Experience", "Packaging", "Overall"]
NAMES = ["Arpita", "Harsh", "Punit"]


def load(name, folder="."):
    d = pd.read_excel(f"{folder}/Annotation_{name}.xlsx", sheet_name="Annotate")
    d = d[d["ID"] != "EXAMPLE"].set_index("ID")
    skip = d["Notes"].astype(str).str.strip().str.lower().eq("skip")
    labels = d[CATS].fillna("None").astype(str).apply(lambda s: s.str.strip().str.capitalize())
    labels = labels.replace({"": "None", "Nan": "None"})
    return labels, skip, d["Review"]


def fleiss(matrix):
    """matrix: rows = items, columns = annotators (labels as strings)."""
    table, _ = aggregate_raters(matrix)
    if table.shape[1] < 2:
        return float("nan")   # everyone gave the same label everywhere
    return fleiss_kappa(table)


def run(folder="."):
    data = {n: load(n, folder) for n in NAMES}
    skipped = set()
    for n in NAMES:
        skipped |= set(data[n][1][data[n][1]].index)
    keep = [i for i in data[NAMES[0]][0].index if i not in skipped]
    L = {n: data[n][0].loc[keep] for n in NAMES}
    texts = data[NAMES[0]][2].loc[keep]
    print(f"Reviews used: {len(keep)}   (skipped by someone: {len(skipped)})\n")

    # 1. Category detection: is the category mentioned or not?
    mention = {n: (L[n] != "None").values.ravel() for n in NAMES}
    # 2. Category + sentiment: the full label (None / Positive / Negative / Neutral)
    full = {n: L[n].values.ravel() for n in NAMES}

    print("=== 1. Pairwise Cohen's kappa ===")
    rows = []
    for a, b in combinations(NAMES, 2):
        rows.append({"Pair": f"{a} vs {b}",
                     "Category detection": round(cohen_kappa_score(mention[a], mention[b]), 3),
                     "Category + sentiment": round(cohen_kappa_score(full[a], full[b]), 3)})
    pair = pd.DataFrame(rows)
    print(pair.to_string(index=False))

    print("\n=== 2. Fleiss' kappa (all 3 annotators together) ===")
    fk_mention = fleiss(np.column_stack([mention[n].astype(int) for n in NAMES]))
    fk_full = fleiss(np.column_stack([full[n] for n in NAMES]))
    print(f"Category detection:   {fk_mention:.3f}")
    print(f"Category + sentiment: {fk_full:.3f}")

    # 3. Sentiment only, where all three agree the category is mentioned
    stacked = np.column_stack([full[n] for n in NAMES])
    both = (stacked != "None").all(axis=1)
    fk_sent = fleiss(stacked[both]) if both.sum() > 1 else float("nan")
    print(f"Sentiment only (on {both.sum()} aspects all 3 found): {fk_sent:.3f}")

    print("\n=== 3. Per category ===")
    per = []
    for c in CATS:
        m = np.column_stack([L[n][c].values for n in NAMES])
        votes = (m != "None").sum(axis=1)
        per.append({"Category": c, "Reviews (majority says mentioned)": int((votes >= 2).sum()),
                    "Fleiss kappa": round(fleiss(m), 3)})
    per = pd.DataFrame(per)
    print(per.to_string(index=False))

    # 4. Disagreements to discuss + majority-vote labels
    dis, final = [], []
    for rid in keep:
        for c in CATS:
            labs = [L[n].loc[rid, c] for n in NAMES]
            counts = pd.Series(labs).value_counts()
            maj = counts.index[0] if counts.iloc[0] >= 2 else "DISCUSS"
            if maj not in ("None",):
                final.append({"ID": rid, "Review": texts.loc[rid], "Category": c, "Majority label": maj})
            if len(set(labs)) > 1:
                row = {"ID": rid, "Review": texts.loc[rid], "Category": c}
                row.update(dict(zip(NAMES, labs)))
                row["Majority"] = maj
                dis.append(row)
    dis = pd.DataFrame(dis)
    final = pd.DataFrame(final)
    print(f"\nCells with disagreement: {len(dis)}   (no majority at all: {(dis['Majority'] == 'DISCUSS').sum() if len(dis) else 0})")

    print("\nHow to read kappa (Landis & Koch, 1977):")
    print("  below 0.20 slight | 0.21-0.40 fair | 0.41-0.60 moderate | 0.61-0.80 substantial | above 0.80 almost perfect")
    return pair, per, dis, final


## Results

In [ ]:
pair, per, dis, final = run('.')

## Export
`disagreements.csv` lists every cell where labels differed, for manual adjudication. `majority_labels.csv` gives the majority-vote label per (review, category).

In [ ]:
dis.to_csv('disagreements.csv', index=False)
final.to_csv('majority_labels.csv', index=False)
files.download('disagreements.csv')
files.download('majority_labels.csv')